In [ ]:
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from mantis.architecture import MantisV2
from mantis.trainer import MantisTrainer

from src.data import PROCESSED
from src.dl_dataset import ApneaDataset, load_cache
from src.eval import evaluate_and_log

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "paris-noah/MantisV2"
SEQ_LEN = 512  # must be multiple of 32

In [ ]:
folds_df = pd.read_csv(PROCESSED / "folds.csv")
test_df  = pd.read_csv(PROCESSED / "test_clean.csv")

train_cache = load_cache(folds_df["record"].unique())
test_cache  = load_cache(test_df["record"].unique())

In [ ]:
def build_mantis_arrays(df, cache, mode: str, context_mins: int, seq_len: int = SEQ_LEN):
    """Pull (N, C, L) windows from dl_cache and interpolate L -> seq_len."""
    ds = ApneaDataset(df, cache, mode=mode, context_mins=context_mins, augment=False)
    X = np.stack([ds[i][0].numpy() for i in range(len(ds))])  # (N, C, L)
    X_t = torch.tensor(X, dtype=torch.float32)
    if X_t.shape[-1] != seq_len:
        X_t = F.interpolate(X_t, size=seq_len, mode="linear", align_corners=False)
    return X_t.numpy()

def extract_zeroshot_embeddings(X: np.ndarray) -> np.ndarray:
    net = MantisV2(device=DEVICE).from_pretrained(MODEL_ID)
    fm = MantisTrainer(device=DEVICE, network=net)
    Z = fm.transform(X)
    del fm, net
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return Z

def run_probe_cv(Z_train, Z_test, folds_df, clf_fn, model_name, split_col="patient_fold", cfg=None):
    """5-fold CV on frozen embeddings + averaged 5-fold probabilities for test (unused until Cell 5)."""
    y = folds_df["label"].to_numpy()
    oof_probs = np.zeros(len(folds_df))
    test_probs_folds = []

    for f in range(5):
        tr = folds_df[split_col] != f
        va = folds_df[split_col] == f
        clf = clf_fn()
        clf.fit(Z_train[tr], y[tr])
        oof_probs[va] = clf.predict_proba(Z_train[va])[:, 1]
        test_probs_folds.append(clf.predict_proba(Z_test)[:, 1])

    metrics = evaluate_and_log(
        folds_df, oof_probs,
        model_name=model_name, split_type=split_col,
        config={"backbone": MODEL_ID, **(cfg or {})},
        use_wandb=True,
    )
    return metrics, oof_probs, np.mean(test_probs_folds, axis=0)


In [ ]:
def run_mantis_cv(
    X_tr, X_te, folds_df,
    model_name: str,
    fine_tuning_type: str = "head",   # "head" (frozen backbone) or "full" (end-to-end)
    split_col: str = "patient_fold",
    num_epochs: int = 15,
    batch_size: int = 128,
    lr: float = 2e-4,
    cfg: dict | None = None,
):
    y = folds_df["label"].to_numpy()
    oof_probs = np.zeros(len(folds_df))
    test_folds = []

    for f in range(5):
        tr = (folds_df[split_col] != f).to_numpy()
        va = (folds_df[split_col] == f).to_numpy()

        net = MantisV2(device=DEVICE).from_pretrained(MODEL_ID)
        fm = MantisTrainer(device=DEVICE, network=net)
        fm.fit(
            X_tr[tr], y[tr],
            fine_tuning_type=fine_tuning_type,
            num_epochs=num_epochs,
            batch_size=batch_size,
            base_learning_rate=lr,
            learning_rate_adjusting=True,
        )
        oof_probs[va] = fm.predict_proba(X_tr[va])[:, 1]
        test_folds.append(fm.predict_proba(X_te)[:, 1])

        del fm, net
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    metrics = evaluate_and_log(
        folds_df, oof_probs,
        model_name=model_name, split_type=split_col,
        config={"backbone": MODEL_ID, "fine_tuning_type": fine_tuning_type, "num_epochs": num_epochs, **(cfg or {})},
        use_wandb=True,
    )
    return metrics, oof_probs, np.mean(test_folds, axis=0)

In [ ]:
X_tr_rredr = build_mantis_arrays(folds_df, train_cache, mode="rr_edr", context_mins=3)
X_te_rredr = build_mantis_arrays(test_df,  test_cache,  mode="rr_edr", context_mins=3)
X_tr_raw   = build_mantis_arrays(folds_df, train_cache, mode="raw_ecg", context_mins=1)
X_te_raw   = build_mantis_arrays(test_df,  test_cache,  mode="raw_ecg", context_mins=1)

# 1. Frozen pretrained backbone + native linear head (RR+EDR 3m)
m_head_rredr, _, te_head_rredr = run_mantis_cv(
    X_tr_rredr, X_te_rredr, folds_df,
    model_name="mantisv2_head_rredr_3m", fine_tuning_type="head", num_epochs=20, lr=1e-3,
    cfg={"mode": "rr_edr", "context_mins": 3},
)

# 2. Frozen pretrained backbone + native linear head (Raw ECG 1m)
m_head_raw, _, te_head_raw = run_mantis_cv(
    X_tr_raw, X_te_raw, folds_df,
    model_name="mantisv2_head_rawecg", fine_tuning_type="head", num_epochs=20, lr=1e-3,
    cfg={"mode": "raw_ecg", "context_mins": 1},
)

# 3. Full end-to-end fine-tuning (RR+EDR 3m)
m_full_rredr, _, te_full_rredr = run_mantis_cv(
    X_tr_rredr, X_te_rredr, folds_df,
    model_name="mantisv2_full_rredr_3m", fine_tuning_type="full", num_epochs=15, lr=2e-4,
    cfg={"mode": "rr_edr", "context_mins": 3},
)

In [ ]:
test_runs = [
    (
        "mantisv2_head_rredr_3m",
        te_head_rredr,
        m_head_rredr["threshold"],
        {"mode": "rr_edr", "context_mins": 3, "fine_tuning_type": "head"},
    ),
    (
        "mantisv2_head_rawecg",
        te_head_raw,
        m_head_raw["threshold"],
        {"mode": "raw_ecg", "context_mins": 1, "fine_tuning_type": "head"},
    ),
    (
        "mantisv2_full_rredr_3m",
        te_full_rredr,
        m_full_rredr["threshold"],
        {"mode": "rr_edr", "context_mins": 3, "fine_tuning_type": "full"},
    ),
]

test_summary = {}
for name, probs, thr, cfg in test_runs:
    test_summary[name] = evaluate_and_log(
        test_df,
        probs,
        model_name=name,
        split_type="test",
        threshold=thr,
        config={"backbone": MODEL_ID, **cfg},
        use_wandb=True,
    )

test_table = pd.DataFrame(test_summary).T.round(4)
display(test_table)